In [1]:
import requests
from bs4 import BeautifulSoup


In [7]:
def fetching_pubmed_articles_with_metadata(query: str, max_results=3, use_mock_if_empty=True):
    headers = {"User-Agent": "Mozilla/5.0"}

    # Step 1: Search PubMed
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    search_params = {
        "db": "pubmed",
        "term": query,
        "retmax": max_results,
        "retmode": "json"
    }
    try:
        search_response = requests.get(search_url, params=search_params, headers=headers, timeout=10).json()

        # --- new validation check goes here ---
        if "esearchresult" not in search_response or "idlist" not in search_response.get("esearchresult", {}):
            print("Unexpected response:", search_response)
            raise ValueError("Malformed PubMed response")
        # ----------------------------------------

        id_list = search_response["esearchresult"]["idlist"]
        print("Found PubMed IDs:", id_list)
        
        if not id_list:
            raise ValueError("No IDs found for this query.")

        ids = ",".join(id_list)

        # Step 2: Fetch article summaries
        fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        fetch_params = {
            "db": "pubmed",
            "id": ids,
            "retmode": "xml"
        }
        fetch_response = requests.get(fetch_url, params=fetch_params, headers=headers, timeout=10)
        soup = BeautifulSoup(fetch_response.text, "lxml")
        articles_xml = soup.find_all("pubmedarticle")
        print("Articles found in XML:", len(articles_xml))

        articles_info = []
        for article, pmid in zip(articles_xml, id_list):
            title_tag = article.find("articletitle")
            abstract_tag = article.find("abstract")
            date_tag = article.find("pubdate")
            author_tags = article.find_all("author")

            # Title
            title = title_tag.get_text(strip=True) if title_tag else "No title"

            # Abstract
            abstract = abstract_tag.get_text(separator=" ", strip=True) if abstract_tag else "No abstract available"

            # Authors
            authors = []
            for author in author_tags:
                last = author.find("lastname")
                fore = author.find("forename")
                if last and fore:
                    authors.append(f"{fore.get_text()} {last.get_text()}")
                elif last:
                    authors.append(last.get_text())
            authors = authors if authors else ["No authors listed"]

            # Publication Date
            pub_date = "No date"
            if date_tag:
                year = date_tag.find("year")
                month = date_tag.find("month")
                pub_date = f"{month.get_text()} {year.get_text()}" if year and month else year.get_text() if year else "No date"

            # PubMed Article URL
            url = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"

            print(f"Article: {title}\n   - Authors: {authors}\n   - Date: {pub_date}\n   - URL: {url}\n")
            articles_info.append({
                "title": title,
                "abstract": abstract,
                "authors": authors,
                "publication_date": pub_date,
                "article_url": url
            })

        if not articles_info and use_mock_if_empty:
            print("No valid articles found, returning mock data.")
            return [{
                "title": "Simulated Study on Fever",
                "abstract": "This is a simulated abstract on the treatment of fever in adults.",
                "authors": ["John Doe", "Jane Smith"],
                "publication_date": "March 2024",
                "article_url": "https://pubmed.ncbi.nlm.nih.gov/12345678/"
            }]
        return articles_info

    except Exception as e:
        print(f"Error during PubMed fetch: {e}")
        if use_mock_if_empty:
            return [{
                "title": "Simulated Study on Fever",
                "abstract": "This is a simulated abstract on the treatment of fever in adults.",
                "authors": ["John Doe", "Jane Smith"],
                "publication_date": "March 2024",
                "article_url": "https://pubmed.ncbi.nlm.nih.gov/12345678/"
            }]
        else:
            return [{"message": f"Error: {e}"}]

In [8]:
fetching_pubmed_articles_with_metadata("fatigue and weakness", max_results=3)

Found PubMed IDs: ['42396406', '42389444', '42387593']
Articles found in XML: 3
Article: Case Report: Subacute combined degeneration misdiagnosed as a primary affective disorder: diagnostic pitfalls and clinical red flags.
   - Authors: ['Yirui Dai', 'Yan Wang', 'Yuanyuan Chen', 'Jinfang Li', 'Xinyu Jia', 'Dongbin Cai', 'Guixiong Zhang', 'Haihong Guo', 'Songjiao Wei', 'Jianjun Wang', 'Xiude Qin']
   - Date: 2026
   - URL: https://pubmed.ncbi.nlm.nih.gov/42396406/

Article: Cold agglutination as a pivotal diagnostic clue for Waldenström macroglobulinemia: A rare case report with diagnostic and therapeutic insights.
   - Authors: ['Wei Zhang', 'Yongwu Xia', 'Zihua Yang', 'Liubing Zhang', 'Xiaoxin Jiang', 'Ting Cai', 'Pinghong Ming']
   - Date: Aug 2026
   - URL: https://pubmed.ncbi.nlm.nih.gov/42389444/

Article: Safety, feasibility and preliminary effects of Atalante exoskeleton-assisted gait training in amyotrophic lateral sclerosis: a prospective ABA pilot study.
   - Authors: ['Ghida

C:\Users\bikra\AppData\Local\Temp\ipykernel_19040\2678619961.py:37: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(fetch_response.text, "lxml")


[{'title': 'Case Report: Subacute combined degeneration misdiagnosed as a primary affective disorder: diagnostic pitfalls and clinical red flags.',
  'abstract': 'Subacute combined degeneration (SCD) of the spinal cord is a progressive neurological disorder caused by vitamin B12 deficiency. When initial symptoms present as nonspecific fatigue, dizziness, and affective distress, the condition is frequently misattributed to primary psychiatric disorders. This "diagnostic overshadowing" leads to critical delays in administering definitive therapy, risking irreversible axonal damage. A 39-year-old male presented with a nine-month history of progressive paresthesia, dizziness, and emotional distress. Initial medical workups (including brain CT and EEG) were unremarkable, leading to a diagnosis of Somatic Symptom Disorder and anxiety. Despite treatment with multiple psychotropic agents, his condition deteriorated, eventually manifesting as lower limb weakness and significant gait ataxia. Sub